# LA Studio Unified Dubbing Coordinator (optional)

This is a real one-URL, one-token coordinator for the selected Dubbing models. It starts each selected **exact CUDA worker** privately, verifies its ordinary `/health`, then exposes it through one Cloudflare tunnel. It never substitutes local CPU or API Gateway routes.

Keep the normal per-model notebooks if you prefer them. This notebook is optional and does not replace those routes.


In [ ]:
import subprocess
from pathlib import Path

SOURCE_REPOSITORY = 'https://github.com/khoinguyen59/kova-video-studio.git'
SOURCE_COMMIT = '96a2fe9'
SOURCE_ROOT = Path('/content/la-studio-unified-source')
subprocess.run(['rm', '-rf', str(SOURCE_ROOT)], check=True)
subprocess.run(['git', 'clone', '--no-checkout', SOURCE_REPOSITORY, str(SOURCE_ROOT)], check=True)
subprocess.run(['git', '-C', str(SOURCE_ROOT), 'checkout', '--detach', SOURCE_COMMIT], check=True)
subprocess.run(['python3', '-m', 'pip', 'install', '--quiet', '--upgrade', 'fastapi==0.115.12', 'uvicorn==0.34.3', 'httpx==0.28.1'], check=True)
print('Pinned LA Studio source:', SOURCE_COMMIT)


In [ ]:
import json
                from pathlib import Path

                # Keep only the exact models you intend to select in LA Studio.
                # Add another supported capability/model pair here before starting the notebook.
                UNIFIED_WORKERS = [
    {
        "capability": "voice-isolation",
        "model": "sherpa-onnx-spleeter-2stems-fp16"
    },
    {
        "capability": "stt",
        "model": "whisper.cpp"
    },
    {
        "capability": "subtitle-ocr",
        "model": "pp-ocrv5-multilingual-3.1"
    },
    {
        "capability": "translation",
        "model": "m2m100-418m"
    },
    {
        "capability": "tts",
        "model": "kokoro"
    },
    {
        "capability": "forced-alignment",
        "model": "mms-forced-aligner-onnx"
    }
]
                CONFIG_PATH = Path('/content/la_studio_unified_workers.json')
                CONFIG_PATH.write_text(json.dumps(UNIFIED_WORKERS, indent=2), encoding='utf-8')
                print('Will prewarm:', ', '.join(f"{row['capability']}/{row['model']}" for row in UNIFIED_WORKERS))


In [ ]:
from pathlib import Path
COORDINATOR_PATH = Path('/content/la_studio_unified_dubbing_coordinator.py')
COORDINATOR_PATH.write_text('#!/usr/bin/env python3\n"""One-tunnel coordinator for real LA Studio Dubbing Colab workers.\n\nThe coordinator deliberately does not implement inference itself.  It starts\nthe selected *exact* notebook workers on private loopback ports, waits for each\nworker\'s real CUDA /health response, and exposes them through one authenticated\nCloudflare URL:\n\n    /v1/unified/<capability>/<model>/<the normal worker route>\n\nThis keeps the direct per-model notebooks valid while allowing the optional\nUnified Dubbing setup in the desktop app to use one URL and token.  A worker\nthat fails to install, load CUDA, or pass its normal health check prevents the\ncoordinator from becoming ready; it is never reported as a successful fake\nroute.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nimport asyncio\nimport json\nimport os\nimport re\nimport secrets\nimport shutil\nimport socket\nimport subprocess\nimport sys\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any\n\nimport httpx\nimport uvicorn\nfrom fastapi import FastAPI, HTTPException, Request\nfrom fastapi.responses import JSONResponse, StreamingResponse\n\n\nCOORDINATOR_REVISION = "unified-dubbing-coordinator-2026-08-22.1"\nLISTEN_HOST = "127.0.0.1"\nLISTEN_PORT = 3960\nTOKEN_ENVIRONMENTS = {\n    "stt": "LA_STUDIO_COLAB_STT_TOKEN",\n    "subtitle-ocr": "LA_STUDIO_COLAB_SUBTITLE_OCR_TOKEN",\n    "translation": "LA_STUDIO_COLAB_TRANSLATION_TOKEN",\n    "tts": "LA_STUDIO_COLAB_TTS_TOKEN",\n    "voice-isolation": "LA_STUDIO_COLAB_SEPARATION_TOKEN",\n    "forced-alignment": "LA_STUDIO_COLAB_ALIGNMENT_TOKEN",\n    "llm": "LA_STUDIO_COLAB_LLM_TOKEN",\n}\nHOP_BY_HOP_HEADERS = {\n    "connection", "keep-alive", "proxy-authenticate", "proxy-authorization",\n    "te", "trailers", "transfer-encoding", "upgrade", "host",\n}\nSAFE_SLUG = re.compile(r"^[a-z0-9][a-z0-9._-]{0,127}$")\n\n\n@dataclass(frozen=True)\nclass WorkerSpec:\n    capability: str\n    model: str\n    notebook: Path\n\n\n@dataclass\nclass RunningWorker:\n    spec: WorkerSpec\n    port: int\n    process: subprocess.Popen[str]\n    log_path: Path\n\n    @property\n    def base_url(self) -> str:\n        return f"http://{LISTEN_HOST}:{self.port}"\n\n\ndef require_slug(value: str, label: str) -> str:\n    if not isinstance(value, str) or not SAFE_SLUG.fullmatch(value):\n        raise ValueError(f"{label} must be a lowercase model/capability slug")\n    return value\n\n\ndef find_free_port() -> int:\n    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as candidate:\n        candidate.bind((LISTEN_HOST, 0))\n        return int(candidate.getsockname()[1])\n\n\ndef read_notebook(path: Path) -> dict[str, Any]:\n    with path.open("r", encoding="utf-8") as stream:\n        return json.load(stream)\n\n\ndef notebook_source(cell: dict[str, Any]) -> str:\n    source = cell.get("source", "")\n    return "".join(source) if isinstance(source, list) else str(source)\n\n\ndef metadata_for_notebook(path: Path) -> tuple[str, str] | None:\n    metadata = read_notebook(path).get("metadata", {}).get("la_studio", {})\n    capability = metadata.get("capability")\n    model = metadata.get("family_id") or metadata.get("model_id")\n    if not isinstance(capability, str) or not isinstance(model, str):\n        return None\n    return capability, model\n\n\ndef discover_exact_notebook(source_root: Path, capability: str, model: str) -> Path:\n    for notebook in sorted((source_root / "notebooks").glob("*.ipynb")):\n        metadata = metadata_for_notebook(notebook)\n        if metadata == (capability, model):\n            return notebook\n    raise RuntimeError(\n        f"No exact generated notebook exists for {capability}/{model}. "\n        "Choose an exact model listed by the Dubbing app, then regenerate this notebook."\n    )\n\n\ndef literal_worker_source(document: dict[str, Any]) -> str | None:\n    """Return a static Path(...).write_text(<worker>) payload from a notebook."""\n    def static_string(node: ast.AST) -> str | None:\n        if isinstance(node, ast.Constant) and isinstance(node.value, str):\n            return node.value\n        if isinstance(node, ast.BinOp) and isinstance(node.op, ast.Add):\n            left = static_string(node.left)\n            right = static_string(node.right)\n            return left + right if left is not None and right is not None else None\n        return None\n\n    for cell in document.get("cells", []):\n        if cell.get("cell_type") != "code":\n            continue\n        try:\n            tree = ast.parse(notebook_source(cell))\n        except SyntaxError:\n            continue\n        for statement in ast.walk(tree):\n            if not isinstance(statement, ast.Call) or not isinstance(statement.func, ast.Attribute):\n                continue\n            if statement.func.attr != "write_text" or not statement.args:\n                continue\n            value = static_string(statement.args[0])\n            if value is not None and "FastAPI" in value:\n                return value\n    return None\n\n\ndef stt_worker_source(document: dict[str, Any]) -> str | None:\n    """The STT generator declares its app directly instead of write_text()."""\n    for cell in document.get("cells", []):\n        source = notebook_source(cell)\n        if "app = FastAPI" in source and "@app.get(\\"/health\\")" in source:\n            return source\n    return None\n\n\ndef worker_source_for(spec: WorkerSpec, source_root: Path) -> str:\n    if spec.capability == "voice-isolation" and spec.model == "sherpa-onnx-spleeter-2stems-fp16":\n        return (source_root / "notebooks" / "workers" /\n                "LA_STUDIO_SEPARATION_SPLEETER_2STEMS_WORKER.py").read_text(encoding="utf-8")\n    document = read_notebook(spec.notebook)\n    source = literal_worker_source(document)\n    if source is None and spec.capability == "stt":\n        source = stt_worker_source(document)\n        if source:\n            source = re.sub(\n                r"TOKEN\\s*=\\s*secrets\\.token_urlsafe\\(32\\)",\n                "TOKEN = os.environ[\'LA_STUDIO_COLAB_STT_TOKEN\']",\n                source,\n            )\n    if source is None:\n        raise RuntimeError(\n            f"Could not extract the exact worker source for {spec.capability}/{spec.model}. "\n            "The notebook must keep a static FastAPI worker source."\n        )\n    return source\n\n\ndef install_shell_lines(document: dict[str, Any], runtime: Path) -> None:\n    """Run only explicit package/artifact setup commands, never notebook launch cells."""\n    for cell in document.get("cells", []):\n        if cell.get("cell_type") != "code":\n            continue\n        source = notebook_source(cell)\n        if "uvicorn" in source and ("cloudflared" in source or "tunnel" in source):\n            continue\n        for raw_line in source.splitlines():\n            line = raw_line.strip()\n            if line.startswith("%pip "):\n                arguments = [sys.executable, "-m", "pip", *line[5:].strip().split()]\n            elif line.startswith("!"):\n                command = line[1:].strip()\n                if command.startswith(("python ", "python3 ")) or "cloudflared" in command:\n                    continue\n                arguments = ["bash", "-lc", command]\n            else:\n                continue\n            subprocess.run(arguments, cwd=runtime, check=True)\n\n\ndef run_ocr_bootstrap(document: dict[str, Any], runtime: Path) -> None:\n    """Use the OCR notebook\'s isolated bootstrap rather than mixing Paddle globally."""\n    for cell in document.get("cells", []):\n        source = notebook_source(cell)\n        if "OCR_SITE_PACKAGES" not in source or "BOOTSTRAP_REVISION" not in source:\n            continue\n        source = source.replace("!nvidia-smi", "subprocess.run([\'nvidia-smi\'], check=True)")\n        bootstrap = runtime / "la_studio_ocr_bootstrap.py"\n        bootstrap.write_text(source, encoding="utf-8")\n        subprocess.run([sys.executable, str(bootstrap)], cwd=runtime, check=True)\n        return\n    raise RuntimeError("The selected Subtitle OCR notebook has no recognized isolated bootstrap cell")\n\n\ndef prepare_worker(spec: WorkerSpec, source_root: Path, runtime: Path) -> tuple[Path, dict[str, str]]:\n    document = read_notebook(spec.notebook)\n    if spec.capability == "subtitle-ocr":\n        run_ocr_bootstrap(document, runtime)\n    else:\n        install_shell_lines(document, runtime)\n    # Uvicorn imports a Python module, so exact model IDs such as\n    # ``whisper.cpp`` and ``m2m100-418m`` cannot be used verbatim as names.\n    module_stem = re.sub(r"[^A-Za-z0-9_]", "_", f"worker_{spec.capability}_{spec.model}")\n    worker_path = runtime / f"{module_stem}.py"\n    worker_path.write_text(worker_source_for(spec, source_root), encoding="utf-8")\n    environment = os.environ.copy()\n    environment["PYTHONUNBUFFERED"] = "1"\n    environment["LA_STUDIO_UNIFIED_DUBBING_TOKEN"] = environment["LA_STUDIO_UNIFIED_DUBBING_TOKEN"]\n    for token_environment in TOKEN_ENVIRONMENTS.values():\n        environment[token_environment] = environment["LA_STUDIO_UNIFIED_DUBBING_TOKEN"]\n    if spec.capability == "subtitle-ocr":\n        isolated_site = runtime / "la_studio_subtitle_ocr_site"\n        environment["PYTHONPATH"] = str(isolated_site)\n        environment["PYTHONNOUSERSITE"] = "1"\n    return worker_path, environment\n\n\ndef health_payload(base_url: str, token: str) -> dict[str, Any] | None:\n    try:\n        response = httpx.get(\n            f"{base_url}/health", headers={"Authorization": f"Bearer {token}"}, timeout=10.0\n        )\n        response.raise_for_status()\n        payload = response.json()\n        return payload if isinstance(payload, dict) else None\n    except (httpx.HTTPError, ValueError):\n        return None\n\n\ndef wait_for_exact_health(worker: RunningWorker, token: str, timeout_seconds: float = 420.0) -> None:\n    deadline = time.monotonic() + timeout_seconds\n    while time.monotonic() < deadline:\n        if worker.process.poll() is not None:\n            tail = worker.log_path.read_text(encoding="utf-8", errors="replace")[-12000:]\n            raise RuntimeError(\n                f"{worker.spec.capability}/{worker.spec.model} exited before readiness "\n                f"(exit {worker.process.returncode}).\\n{tail}"\n            )\n        payload = health_payload(worker.base_url, token)\n        if payload and payload.get("ready") is True:\n            returned_model = str(payload.get("model") or payload.get("family_id") or "")\n            if returned_model and returned_model != worker.spec.model:\n                raise RuntimeError(\n                    f"Worker identity mismatch: expected {worker.spec.model}, got {returned_model}"\n                )\n            if str(payload.get("device", "cuda")).lower() != "cuda":\n                raise RuntimeError(\n                    f"{worker.spec.capability}/{worker.spec.model} is not CUDA-ready: {payload}"\n                )\n            return\n        time.sleep(1.0)\n    raise RuntimeError(\n        f"Timed out waiting for actual CUDA health from {worker.spec.capability}/{worker.spec.model}. "\n        f"See {worker.log_path}."\n    )\n\n\nclass UnifiedCoordinator:\n    def __init__(self, source_root: Path, runtime: Path, token: str):\n        self.source_root = source_root\n        self.runtime = runtime\n        self.token = token\n        self.workers: dict[tuple[str, str], RunningWorker] = {}\n\n    def start(self, selections: list[dict[str, Any]]) -> None:\n        if not selections:\n            raise RuntimeError("UNIFIED_WORKERS is empty; configure at least one exact Dubbing model")\n        self.runtime.mkdir(parents=True, exist_ok=True)\n        for selection in selections:\n            capability = require_slug(selection.get("capability"), "capability")\n            model = require_slug(selection.get("model"), "model")\n            key = (capability, model)\n            if key in self.workers:\n                continue\n            notebook = discover_exact_notebook(self.source_root, capability, model)\n            spec = WorkerSpec(capability, model, notebook)\n            worker_path, environment = prepare_worker(spec, self.source_root, self.runtime)\n            port = find_free_port()\n            log_path = self.runtime / f"{capability}-{model}.log"\n            with log_path.open("w", encoding="utf-8") as log:\n                process = subprocess.Popen(\n                    [sys.executable, "-m", "uvicorn", f"{worker_path.stem}:app",\n                     "--host", LISTEN_HOST, "--port", str(port)],\n                    cwd=self.runtime, env=environment, stdout=log, stderr=subprocess.STDOUT, text=True,\n                )\n            worker = RunningWorker(spec, port, process, log_path)\n            wait_for_exact_health(worker, self.token)\n            self.workers[key] = worker\n\n    def worker_for(self, capability: str, model: str) -> RunningWorker:\n        worker = self.workers.get((capability, model))\n        if worker is None:\n            raise HTTPException(\n                status_code=404,\n                detail=(f"{capability}/{model} was not prewarmed by this unified notebook. "\n                        "Add that exact model to UNIFIED_WORKERS and run the notebook again."),\n            )\n        return worker\n\n    def health(self) -> dict[str, Any]:\n        result: list[dict[str, Any]] = []\n        for worker in self.workers.values():\n            payload = health_payload(worker.base_url, self.token)\n            if not payload or payload.get("ready") is not True:\n                raise HTTPException(status_code=503, detail=f"Worker lost readiness: {worker.spec}")\n            result.append({"capability": worker.spec.capability, "model": worker.spec.model, "health": payload})\n        return {"ready": True, "coordinator": COORDINATOR_REVISION, "workers": result}\n\n    def stop(self) -> None:\n        for worker in self.workers.values():\n            if worker.process.poll() is None:\n                worker.process.terminate()\n\n\nCOORDINATOR: UnifiedCoordinator | None = None\nAPP = FastAPI(title="LA Studio unified Dubbing coordinator")\n\n\ndef require_authorization(request: Request) -> None:\n    expected = f"Bearer {os.environ[\'LA_STUDIO_UNIFIED_DUBBING_TOKEN\']}"\n    if not secrets.compare_digest(request.headers.get("authorization", ""), expected):\n        raise HTTPException(status_code=401, detail="Invalid LA Studio unified session token")\n\n\n@APP.get("/health")\nasync def coordinator_health(request: Request) -> JSONResponse:\n    require_authorization(request)\n    if COORDINATOR is None:\n        raise HTTPException(status_code=503, detail="Coordinator has not finished prewarming exact workers")\n    return JSONResponse(COORDINATOR.health())\n\n\n@APP.get("/v1/capabilities")\nasync def capabilities(request: Request) -> JSONResponse:\n    require_authorization(request)\n    if COORDINATOR is None:\n        raise HTTPException(status_code=503, detail="Coordinator has not finished prewarming exact workers")\n    rows = [{"capability": worker.spec.capability, "model": worker.spec.model}\n            for worker in COORDINATOR.workers.values()]\n    return JSONResponse({"ready": True, "routes": rows})\n\n\n@APP.api_route("/v1/unified/{capability}/{model}/{route:path}", methods=["GET", "POST", "PUT", "PATCH", "DELETE", "HEAD", "OPTIONS"])\nasync def proxy(capability: str, model: str, route: str, request: Request) -> StreamingResponse:\n    require_authorization(request)\n    if COORDINATOR is None:\n        raise HTTPException(status_code=503, detail="Coordinator has not finished prewarming exact workers")\n    worker = COORDINATOR.worker_for(require_slug(capability, "capability"), require_slug(model, "model"))\n    target = f"{worker.base_url}/{route.lstrip(\'/\')}"\n    if request.url.query:\n        target += f"?{request.url.query}"\n    headers = {key: value for key, value in request.headers.items() if key.lower() not in HOP_BY_HOP_HEADERS}\n    client = httpx.AsyncClient(timeout=httpx.Timeout(connect=30.0, read=None, write=None, pool=30.0))\n    try:\n        upstream_request = client.build_request(request.method, target, headers=headers, content=request.stream())\n        upstream = await client.send(upstream_request, stream=True)\n    except httpx.HTTPError as error:\n        await client.aclose()\n        raise HTTPException(status_code=502, detail=f"Configured unified worker request failed: {error}") from error\n\n    async def response_body():\n        try:\n            async for chunk in upstream.aiter_raw():\n                yield chunk\n        finally:\n            await upstream.aclose()\n            await client.aclose()\n\n    response_headers = {key: value for key, value in upstream.headers.items() if key.lower() not in HOP_BY_HOP_HEADERS}\n    return StreamingResponse(response_body(), status_code=upstream.status_code, headers=response_headers)\n\n\ndef ensure_cloudflared(runtime: Path) -> str:\n    found = shutil.which("cloudflared")\n    if found:\n        return found\n    destination = runtime / "cloudflared"\n    subprocess.run([\n        "curl", "--fail", "--location", "--retry", "3", "--output", str(destination),\n        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",\n    ], check=True)\n    destination.chmod(0o755)\n    return str(destination)\n\n\ndef start_tunnel(runtime: Path) -> tuple[subprocess.Popen[str], str]:\n    cloudflared = ensure_cloudflared(runtime)\n    log_path = runtime / "unified-tunnel.log"\n    log = log_path.open("w", encoding="utf-8")\n    tunnel = subprocess.Popen(\n        [cloudflared, "tunnel", "--url", f"http://{LISTEN_HOST}:{LISTEN_PORT}", "--no-autoupdate"],\n        stdout=log, stderr=subprocess.STDOUT, text=True,\n    )\n    pattern = re.compile(r"https://[-a-z0-9]+\\.trycloudflare\\.com", re.IGNORECASE)\n    deadline = time.monotonic() + 90.0\n    while time.monotonic() < deadline:\n        if tunnel.poll() is not None:\n            raise RuntimeError(f"Cloudflare tunnel exited early:\\n{log_path.read_text(encoding=\'utf-8\', errors=\'replace\')[-8000:]}")\n        text = log_path.read_text(encoding="utf-8", errors="replace")\n        match = pattern.search(text)\n        if match:\n            return tunnel, match.group(0)\n        time.sleep(0.5)\n    tunnel.terminate()\n    raise RuntimeError("Timed out waiting for the verified public Cloudflare tunnel URL")\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--source-root", required=True, type=Path)\n    parser.add_argument("--config", required=True, type=Path)\n    parser.add_argument("--runtime", default=Path("/content/la_studio_unified_dubbing"), type=Path)\n    arguments = parser.parse_args()\n    if not arguments.source_root.is_dir():\n        raise RuntimeError(f"Source checkout does not exist: {arguments.source_root}")\n    selections = json.loads(arguments.config.read_text(encoding="utf-8"))\n    if not isinstance(selections, list):\n        raise RuntimeError("Unified workers config must be a JSON list")\n    os.environ.setdefault("LA_STUDIO_UNIFIED_DUBBING_TOKEN", secrets.token_urlsafe(32))\n    global COORDINATOR\n    COORDINATOR = UnifiedCoordinator(arguments.source_root, arguments.runtime, os.environ["LA_STUDIO_UNIFIED_DUBBING_TOKEN"])\n    try:\n        COORDINATOR.start(selections)\n        tunnel, public_url = start_tunnel(arguments.runtime)\n        print("\\nLA Studio Unified Dubbing coordinator is ready.")\n        print(f"LA_STUDIO_UNIFIED_DUBBING_URL={public_url}")\n        print(f"LA_STUDIO_UNIFIED_DUBBING_TOKEN={os.environ[\'LA_STUDIO_UNIFIED_DUBBING_TOKEN\']}")\n        print("Paste these once in Dubbing > Project setup > Unified Colab (optional).")\n        uvicorn.run(APP, host=LISTEN_HOST, port=LISTEN_PORT, log_level="info")\n        tunnel.terminate()\n    finally:\n        if COORDINATOR is not None:\n            COORDINATOR.stop()\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('Wrote coordinator:', COORDINATOR_PATH)


In [ ]:
# This cell remains running while LA Studio uses the unified worker.
# It prints one URL and one token only after every selected exact CUDA worker is healthy.
!python3 /content/la_studio_unified_dubbing_coordinator.py \
    --source-root /content/la-studio-unified-source \
    --config /content/la_studio_unified_workers.json
